# Feature Engineering
## Dataset: Properatti
**Objetivo:** Crear y transformar variables para mejorar el rendimiento del modelo de Machine Learning.

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset preparado
df = pd.read_csv('properatti_preparado.csv')

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

Filas: 79272
Columnas: 6


,property_type,place_name,state_name,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2
0,PH,Mataderos,Capital Federal,62000.0,55.0,40.0
1,apartment,La Plata,Bs.As. G.B.A. Zona Sur,150000.0,80.0,72.0
2,apartment,Mataderos,Capital Federal,72000.0,55.0,55.0
3,PH,Liniers,Capital Federal,95000.0,80.0,72.0
4,apartment,Centro,Buenos Aires Costa Atlántica,64000.0,35.0,35.0


### 1. Creación de nuevas variables

In [ ]:
# 1. Precio promedio por state_name
precio_por_zona = df.groupby('state_name')['price_aprox_usd'].mean()
df['precio_promedio_zona'] = df['state_name'].map(precio_por_zona)

# 2. Precio promedio por tipo de propiedad
precio_por_tipo = df.groupby('property_type')['price_aprox_usd'].mean()
df['precio_promedio_tipo'] = df['property_type'].map(precio_por_tipo)

# 3. Ratio entre superficie cubierta y total
df['ratio_superficie'] = df['surface_covered_in_m2'] / df['surface_total_in_m2']

print("Nuevas variables creadas")
df[['precio_promedio_zona', 'precio_promedio_tipo', 'ratio_superficie']].describe()

Nuevas variables creadas ✓


,precio_promedio_zona,precio_promedio_tipo,ratio_superficie
count,79272.000000,79272.000000,79272.00
mean,180796.648312,180796.648312,inf
std,31163.933770,41236.817921,NaN
min,49000.000000,138967.880813,0.00
25%,168665.827623,151935.533469,0.75
50%,179231.982107,151935.533469,0.90
75%,195662.900763,238794.476777,1.00
max,296500.000000,238794.476777,inf


In [3]:
# Verificar cuántas filas tienen surface_total_in_m2 = 0
print(f"Filas con surface_total = 0: {(df['surface_total_in_m2'] == 0).sum()}")
print(f"Filas con surface_covered = 0: {(df['surface_covered_in_m2'] == 0).sum()}")

# Eliminar filas con superficie = 0
df = df[(df['surface_total_in_m2'] > 0) & (df['surface_covered_in_m2'] > 0)]

# Recalcular ratio
df['ratio_superficie'] = df['surface_covered_in_m2'] / df['surface_total_in_m2']

print(f"\nFilas después de limpieza: {df.shape[0]}")
print(f"\nEstadísticas ratio_superficie:")
print(df['ratio_superficie'].describe())

Filas con surface_total = 0: 185
Filas con surface_covered = 0: 2

Filas después de limpieza: 79085

Estadísticas ratio_superficie:
count    79085.000000
mean         1.029110
std          3.961829
min          0.000400
25%          0.750000
50%          0.900000
75%          1.000000
max        942.307692
Name: ratio_superficie, dtype: float64


In [4]:
# El ratio no puede ser mayor a 1 (superficie cubierta <= superficie total)
df = df[df['ratio_superficie'] <= 1]

print(f"Filas después de limpieza: {df.shape[0]}")
print(f"\nEstadísticas ratio_superficie:")
print(df['ratio_superficie'].describe())

Filas después de limpieza: 67072

Estadísticas ratio_superficie:
count    67072.000000
mean         0.789564
std          0.238071
min          0.000400
25%          0.681818
50%          0.893269
75%          0.975000
max          1.000000
Name: ratio_superficie, dtype: float64


### 2. Encoding de variables categóricas

In [5]:
# One Hot Encoding de property_type y state_name
# place_name tiene demasiadas categorías únicas, usaremos state_name
print(f"Categorías únicas en property_type: {df['property_type'].nunique()}")
print(f"Categorías únicas en state_name: {df['state_name'].nunique()}")
print(f"Categorías únicas en place_name: {df['place_name'].nunique()}")

Categorías únicas en property_type: 4
Categorías únicas en state_name: 24
Categorías únicas en place_name: 788


In [6]:
# Eliminar place_name por tener demasiadas categorías
df = df.drop(columns=['place_name'])

# Aplicar One Hot Encoding a property_type y state_name
df = pd.get_dummies(df, columns=['property_type', 'state_name'], drop_first=False)

print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas creadas: {df.columns.tolist()}")

Filas: 67072
Columnas: 34

Columnas creadas: ['price_aprox_usd', 'surface_total_in_m2', 'surface_covered_in_m2', 'precio_promedio_zona', 'precio_promedio_tipo', 'ratio_superficie', 'property_type_PH', 'property_type_apartment', 'property_type_house', 'property_type_store', 'state_name_Bs.As. G.B.A. Zona Norte', 'state_name_Bs.As. G.B.A. Zona Oeste', 'state_name_Bs.As. G.B.A. Zona Sur', 'state_name_Buenos Aires Costa Atlántica', 'state_name_Buenos Aires Interior', 'state_name_Capital Federal', 'state_name_Catamarca', 'state_name_Chaco', 'state_name_Chubut', 'state_name_Corrientes', 'state_name_Córdoba', 'state_name_Entre Ríos', 'state_name_La Pampa', 'state_name_Mendoza', 'state_name_Misiones', 'state_name_Neuquén', 'state_name_Río Negro', 'state_name_Salta', 'state_name_San Luis', 'state_name_Santa Cruz', 'state_name_Santa Fe', 'state_name_Santiago Del Estero', 'state_name_Tierra Del Fuego', 'state_name_Tucumán']


In [8]:
# Guardar dataset con feature engineering
df.to_csv('properatti_fe.csv', index=False)
print("Dataset guardado correctamente ✓")
print(f"\nResumen final:")
print(f"Filas: {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")
df.head()

Dataset guardado correctamente ✓

Resumen final:
Filas: 67072
Columnas: 34


,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,precio_promedio_zona,precio_promedio_tipo,ratio_superficie,property_type_PH,property_type_apartment,property_type_house,property_type_store,...,state_name_Misiones,state_name_Neuquén,state_name_Río Negro,state_name_Salta,state_name_San Luis,state_name_Santa Cruz,state_name_Santa Fe,state_name_Santiago Del Estero,state_name_Tierra Del Fuego,state_name_Tucumán
0,62000.0,55.0,40.0,179231.982107,138967.880813,0.727273,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,150000.0,80.0,72.0,168665.827623,151935.533469,0.900000,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
2,72000.0,55.0,55.0,179231.982107,151935.533469,1.000000,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
3,95000.0,80.0,72.0,179231.982107,138967.880813,0.900000,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,64000.0,35.0,35.0,133634.905788,151935.533469,1.000000,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False
